In [32]:
import os
import random
import hashlib

In [33]:
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F  # secp256k1 prime
a = 0                                                                    # curve param a
b = 7                                                                    # curve param b
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798  # generator X
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8  # generator Y
G = (Gx, Gy)                                                             # generator point
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141  # curve order


### ECC MATH

In [34]:
def point_add(P, Q):
    """Add two points on the elliptic curve."""
    if P is None: return Q
    if Q is None: return P

    x1, y1 = P
    x2, y2 = Q

    if x1 == x2:
        if y1 != y2:
            return None                       # P + (-P) = point at infinity
        # Point doubling
        lam = (3 * x1 * x1 + a) * pow(2 * y1, -1, p) % p
    else:
        # Standard addition
        lam = (y2 - y1) * pow(x2 - x1, -1, p) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)


def scalar_mult(k, P):
    """Multiply point P by scalar k using double-and-add."""
    result = None   # point at infinity (identity element)
    addend = P

    while k:
        if k & 1:
            result = point_add(result, addend)
        addend = point_add(addend, addend)   # double
        k >>= 1

    return result


# ── Key generation ───────────────────────────────────────────────────────────

def generate_keypair():
    """Generate a private key and its corresponding public key."""
    private_key = random.randrange(2, n)        # random integer in [2, n-1]
    public_key  = scalar_mult(private_key, G)   # public_key = private_key × G
    return private_key, public_key


# ── ECDH key exchange ────────────────────────────────────────────────────────

def derive_shared_secret(private_key, other_public_key):
    """
    Alice computes: alice_priv × Bob_pub   =  alice_priv × bob_priv × G
    Bob computes:    bob_priv × Alice_pub  =  bob_priv × alice_priv × G
    Same result — that's the magic!
    """
    shared_point = scalar_mult(private_key, other_public_key)
    return shared_point


def derive_key(shared_point, length=32):
    """Hash the shared point's x-coordinate into a symmetric key."""
    x_bytes = shared_point[0].to_bytes(32, 'big')
    return hashlib.sha256(x_bytes).digest()[:length]

### ECDH Key Exchange

In [35]:
# 1. Each party generates their key pair independently
alice_priv, alice_pub = generate_keypair()
bob_priv,   bob_pub   = generate_keypair()

print(f"\n[Alice] Private key : {hex(alice_priv)[:18]}…")
print(f"[Alice] Public  key : ({hex(alice_pub[0])[:10]}…, {hex(alice_pub[1])[:10]}…)")
print(f"\n[Bob]   Private key : {hex(bob_priv)[:18]}…")
print(f"[Bob]   Public  key : ({hex(bob_pub[0])[:10]}…, {hex(bob_pub[1])[:10]}…)")


[Alice] Private key : 0x248f15ed542c9f17…
[Alice] Public  key : (0x57c6d53d…, 0x344a72ad…)

[Bob]   Private key : 0x62713d664e892214…
[Bob]   Public  key : (0xe87bb700…, 0x4e760e43…)


In [36]:
alice_shared = derive_shared_secret(alice_priv, bob_pub)
bob_shared   = derive_shared_secret(bob_priv,   alice_pub)

print(f"\n[Alice] Computes: alice_priv × bob_pub  = {hex(alice_shared[0])[:18]}…")
print(f"[Bob]   Computes: bob_priv  × alice_pub = {hex(bob_shared[0])[:18]}…")

if alice_shared == bob_shared:
    print("\nKEY MATCHED!!!")
else:
    print("\nKEY FAILED")


[Alice] Computes: alice_priv × bob_pub  = 0xef9df2615da6c6af…
[Bob]   Computes: bob_priv  × alice_pub = 0xef9df2615da6c6af…

KEY MATCHED!!!
